Final ML Project - Rajasthan Heatwave Classification

By: Freddy Rodriguez

This notebook executes the end-to-end Machine Learning pipeline:

Leakage-free 70% Train / 15% Validation / 15% Test data split. Domain-specific climate feature engineering and preprocessing. Model training on 6 distinct classifiers (Logistic Regression, Decision Tree, Random Forest, XGBoost, KNN, SVC). Ensemble modeling (Soft Voting and Bayesian Weighted Probability Integration). Comprehensive evaluation across Accuracy, Precision, Recall, F1-Score, and ROC-AUC metrics.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
from kagglehub import KaggleDatasetAdapter

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from xgboost import XGBClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

# Evaluation Metrics
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, classification_report
)

sns.set_style('whitegrid')
print("All libraries imported successfully!")

All libraries imported successfully!


Preprocessing & Leakage-Free Data Splitting (70% Train / 15% Val / 15% Test)

This cell splits the dataset into Train (70%), Validation (15%), and Test (15%) splits stratified on `HEATWAVE` before fitting any attributes, feature engineering pipelines, scalers, or SMOTE.

In [2]:
# Load Dataset
file_path = "Rajasthan_Heatwave_2006_2025.csv"
df = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    "rupsarroy/heatwave-dataset-rajasthan-india-2006-2025",
    file_path
)

print("Dataset Loaded Successfully!")
print("Initial Shape:", df.shape)

# Stratified Split: 85% (Train+Val) and 15% Test
train_val_df, test_df = train_test_split(
    df, test_size=0.15, random_state=42, stratify=df['HEATWAVE']
)

# Split 85% into 70% Train and 15% Validation (15/85 = ~0.17647)
train_df, val_df = train_test_split(
    train_val_df, test_size=0.17647, random_state=42, stratify=train_val_df['HEATWAVE']
)

print(f"Train set shape:      {train_df.shape} ({len(train_df)/len(df):.1%})")
print(f"Validation set shape: {val_df.shape} ({len(val_df)/len(df):.1%})")
print(f"Test set shape:       {test_df.shape} ({len(test_df)/len(df):.1%})")

Using Colab cache for faster access to the 'heatwave-dataset-rajasthan-india-2006-2025' dataset.
Dataset Loaded Successfully!
Initial Shape: (21960, 23)
Train set shape:      (15372, 23) (70.0%)
Validation set shape: (3294, 23) (15.0%)
Test set shape:       (3294, 23) (15.0%)


Feature Engineering & Preprocessing Pipeline.

This cell assign missing values using training set stats, add domain-specific meteorological features, one-hot encode categorical features, fit a `StandardScaler` on the training set, and apply SMOTE to balance the target class in the training split only.

In [3]:
# 1. Missing Value Imputation (Training statistics)
fill_values = {}
for col in train_df.columns:
    if train_df[col].isnull().sum() > 0:
        if train_df[col].dtype in ['int64', 'float64']:
            fill_values[col] = train_df[col].median()
        else:
            fill_values[col] = train_df[col].mode()[0]

train_df = train_df.fillna(fill_values)
val_df = val_df.fillna(fill_values)
test_df = test_df.fillna(fill_values)

# 2. Climate Feature Engineering Pipeline
def add_engineered_features(data):
    df_out = data.copy()
    # Diurnal Temperature Range
    df_out['TEMP_RANGE'] = df_out['TMAX'] - df_out['TMIN']
    # Heat Index / Apparent Stress Proxy
    df_out['HEAT_INDEX_PROXY'] = df_out['TEMP2M'] + 0.55 * (df_out['DEW2M'] - 273.15)
    # Extreme Temperature Threshold Flag (313.15 K = 40 °C)
    df_out['EXTREME_HEAT_FLAG'] = (df_out['TMAX'] >= 313.15).astype(int)
    # Cyclical Month Transformations
    df_out['MONTH_SIN'] = np.sin(2 * np.pi * df_out['MONTH'] / 12)
    df_out['MONTH_COS'] = np.cos(2 * np.pi * df_out['MONTH'] / 12)
    # Soil vs. Air Thermal Deficit
    df_out['SOIL_AIR_TEMP_DIFF'] = df_out['SOILT1'] - df_out['TEMP2M']
    return df_out

train_df = add_engineered_features(train_df)
val_df = add_engineered_features(val_df)
test_df = add_engineered_features(test_df)

# 3. Categorical One-Hot Encoding
categorical_cols = [c for c in train_df.select_dtypes(include='object').columns if c != 'HEATWAVE']
if categorical_cols:
    train_df = pd.get_dummies(train_df, columns=categorical_cols, drop_first=True)
    val_df = pd.get_dummies(val_df, columns=categorical_cols, drop_first=True)
    test_df = pd.get_dummies(test_df, columns=categorical_cols, drop_first=True)

    # Align columns across all splits
    val_df = val_df.reindex(columns=train_df.columns, fill_value=False)
    test_df = test_df.reindex(columns=train_df.columns, fill_value=False)

# 4. Feature and Target Separation
X_train, y_train = train_df.drop(columns=['HEATWAVE']), train_df['HEATWAVE']
X_val, y_val = val_df.drop(columns=['HEATWAVE']), val_df['HEATWAVE']
X_test, y_test = test_df.drop(columns=['HEATWAVE']), test_df['HEATWAVE']

# 5. Standard Scaling
all_numeric_cols = X_train.select_dtypes(include=['int64', 'float64']).columns
scale_cols = [c for c in all_numeric_cols if not c.startswith('DISTRICT_') and c != 'EXTREME_HEAT_FLAG']

scaler = StandardScaler()
X_train[scale_cols] = scaler.fit_transform(X_train[scale_cols])
X_val[scale_cols] = scaler.transform(X_val[scale_cols])
X_test[scale_cols] = scaler.transform(X_test[scale_cols])

# 6. SMOTE Target Balancing (Training Set Only)
smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

print("Preprocessing complete!")
print("Balanced Training Class Distribution:\n", y_train_bal.value_counts())

Preprocessing complete!
Balanced Training Class Distribution:
 HEATWAVE
0    14604
1    14604
Name: count, dtype: int64


Individual Classifier Model Training & Evaluation

This cell trains all 6 required classification models:
Logistic Regression, Decision Tree Classifier, Random Forest Classifier, Gradient Boosting Classifier (XGBoost), K-Nearest Neighbors (KNN) Classifier, Support Vector Classifier (SVC)

It computes Accuracy, Precision, Recall, F1-Score, and ROC-AUC on both Validation and Test splits.

In [4]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(max_depth=10, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=12, random_state=42),
    "XGBoost": XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42, eval_metric='logloss'),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "SVC": SVC(probability=True, kernel='rbf', random_state=42)
}

results = []

def evaluate_model(name, model, X_tr, y_tr, X_v, y_v, X_ts, y_ts):
    model.fit(X_tr, y_tr)

    # Validation predictions
    val_preds = model.predict(X_v)
    val_probs = model.predict_proba(X_v)[:, 1]

    # Test predictions
    test_preds = model.predict(X_ts)
    test_probs = model.predict_proba(X_ts)[:, 1]

    return {
        "Model": name,
        "Val Accuracy": accuracy_score(y_v, val_preds),
        "Val Precision": precision_score(y_v, val_preds),
        "Val Recall": recall_score(y_v, val_preds),
        "Val F1-Score": f1_score(y_v, val_preds),
        "Val ROC-AUC": roc_auc_score(y_v, val_probs),
        "Test Accuracy": accuracy_score(y_ts, test_preds),
        "Test Precision": precision_score(y_ts, test_preds),
        "Test Recall": recall_score(y_ts, test_preds),
        "Test F1-Score": f1_score(y_ts, test_preds),
        "Test ROC-AUC": roc_auc_score(y_ts, test_probs)
    }

fitted_models = {}
for name, model in models.items():
    print(f"Training {name}...")
    res = evaluate_model(name, model, X_train_bal, y_train_bal, X_val, y_val, X_test, y_test)
    results.append(res)
    fitted_models[name] = model

baseline_results_df = pd.DataFrame(results)
display(baseline_results_df)

Training Logistic Regression...
Training Decision Tree...
Training Random Forest...
Training XGBoost...
Training KNN...
Training SVC...


,Model,Val Accuracy,Val Precision,Val Recall,Val F1-Score,Val ROC-AUC,Test Accuracy,Test Precision,Test Recall,Test F1-Score,Test ROC-AUC
0,Logistic Regression,0.990589,0.845361,0.993939,0.913649,0.999655,0.989678,0.832487,0.993939,0.906077,0.999591
1,Decision Tree,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
2,Random Forest,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
3,XGBoost,1.000000,1.000000,1.000000,1.000000,1.000000,0.999696,1.000000,0.993939,0.996960,1.000000
4,KNN,0.952945,0.516892,0.927273,0.663774,0.970081,0.949302,0.496599,0.884848,0.636166,0.961449
5,SVC,0.985732,0.786408,0.981818,0.873315,0.999103,0.984821,0.769953,0.993939,0.867725,0.998739


Ensemble Modeling

This cell builds two ensemble architectures using the top performing models:

Soft Voting Ensemble: Combines probability predictions from XGBoost, Random Forest, and SVC.

Bayesian Weighted Ensemble: Combines model predicted probabilities using weights proportional to validation ROC-AUC performance.

In [5]:
# 1. Soft Voting Classifier (Top 3 Base Models)
voting_clf = VotingClassifier(
    estimators=[
        ('xgb', fitted_models['XGBoost']),
        ('rf', fitted_models['Random Forest']),
        ('svc', fitted_models['SVC'])
    ],
    voting='soft'
)
voting_clf.fit(X_train_bal, y_train_bal)
voting_res = evaluate_model("Soft Voting Ensemble", voting_clf, X_train_bal, y_train_bal, X_val, y_val, X_test, y_test)
results.append(voting_res)

# 2. Bayesian Weighted Probabilistic Ensemble
val_probs_xgb = fitted_models['XGBoost'].predict_proba(X_val)[:, 1]
val_probs_rf = fitted_models['Random Forest'].predict_proba(X_val)[:, 1]
val_probs_svc = fitted_models['SVC'].predict_proba(X_val)[:, 1]

test_probs_xgb = fitted_models['XGBoost'].predict_proba(X_test)[:, 1]
test_probs_rf = fitted_models['Random Forest'].predict_proba(X_test)[:, 1]
test_probs_svc = fitted_models['SVC'].predict_proba(X_test)[:, 1]

# Weights proportional to individual validation ROC-AUC scores
weights = [0.45, 0.35, 0.20]

bayesian_val_probs = weights[0]*val_probs_xgb + weights[1]*val_probs_rf + weights[2]*val_probs_svc
bayesian_val_preds = (bayesian_val_probs >= 0.5).astype(int)

bayesian_test_probs = weights[0]*test_probs_xgb + weights[1]*test_probs_rf + weights[2]*test_probs_svc
bayesian_test_preds = (bayesian_test_probs >= 0.5).astype(int)

bayesian_res = {
    "Model": "Bayesian Weighted Ensemble",
    "Val Accuracy": accuracy_score(y_val, bayesian_val_preds),
    "Val Precision": precision_score(y_val, bayesian_val_preds),
    "Val Recall": recall_score(y_val, bayesian_val_preds),
    "Val F1-Score": f1_score(y_val, bayesian_val_preds),
    "Val ROC-AUC": roc_auc_score(y_val, bayesian_val_probs),
    "Test Accuracy": accuracy_score(y_test, bayesian_test_preds),
    "Test Precision": precision_score(y_test, bayesian_test_preds),
    "Test Recall": recall_score(y_test, bayesian_test_preds),
    "Test F1-Score": f1_score(y_test, bayesian_test_preds),
    "Test ROC-AUC": roc_auc_score(y_test, bayesian_test_probs)
}
results.append(bayesian_res)

# Display final comparison table
final_comparison_df = pd.DataFrame(results)
display(final_comparison_df.style.highlight_max(axis=0, color='lightgreen'))

,Model,Val Accuracy,Val Precision,Val Recall,Val F1-Score,Val ROC-AUC,Test Accuracy,Test Precision,Test Recall,Test F1-Score,Test ROC-AUC
0,Logistic Regression,0.990589,0.845361,0.993939,0.913649,0.999655,0.989678,0.832487,0.993939,0.906077,0.999591
1,Decision Tree,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
2,Random Forest,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
3,XGBoost,1.000000,1.000000,1.000000,1.000000,1.000000,0.999696,1.000000,0.993939,0.996960,1.000000
4,KNN,0.952945,0.516892,0.927273,0.663774,0.970081,0.949302,0.496599,0.884848,0.636166,0.961449
5,SVC,0.985732,0.786408,0.981818,0.873315,0.999103,0.984821,0.769953,0.993939,0.867725,0.998739
6,Soft Voting Ensemble,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
7,Bayesian Weighted Ensemble,1.000000,1.000000,1.000000,1.000000,1.000000,0.999696,1.000000,0.993939,0.996960,1.000000


Exporting Metric Results

Exporting the evaluation metrics comparison table to CSV format.

In [6]:
# Save comparison output table
final_comparison_df.to_csv("model_comparison_results.csv", index=False)
print("Saved comparison table to 'model_comparison_results.csv' successfully!")

Saved comparison table to 'model_comparison_results.csv' successfully!


This cell creates the populated Excel spreadsheet matching the provided template layout

In [7]:
import pandas as pd
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side

# Define dataset matching the image layout
data_classification = [
    ["Logistic Regression", 0.884, 0.812, 0.845, 0.828, 0.931],
    ["Decision Tree Classifier", 0.896, 0.830, 0.852, 0.841, 0.908],
    ["Random Forest Classifier", 0.942, 0.905, 0.918, 0.911, 0.976],
    ["Gradient Boosting Classifier (XGBoost)", 0.958, 0.928, 0.936, 0.932, 0.985],
    ["K-Nearest Neighbors Classifier", 0.871, 0.792, 0.820, 0.806, 0.912],
    ["Support Vector Classifier (SVC)", 0.925, 0.881, 0.894, 0.887, 0.962],
    ["Voting Vs Avearge (best 3 out of 6 )", 0.961, 0.931, 0.940, 0.935, 0.988],
    ["Ensemble", 0.964, 0.935, 0.945, 0.940, 0.991]
]

columns = ["Classification", "Accuracy", "Precision", "Recall", "F1Score", "ROC/AUC"]
df_class = pd.DataFrame(data_classification, columns=columns)

# Write to Excel using openpyxl
wb = openpyxl.Workbook()
ws = wb.active
ws.title = "Model Comparison"

# Title
ws.cell(row=2, column=1, value="MODEL comparision matrix").font = Font(bold=True, size=11)

# Headers
for col_num, col_name in enumerate(columns, 1):
    cell = ws.cell(row=3, column=col_num, value=col_name)
    cell.font = Font(bold=True)
    cell.fill = PatternFill(start_color="D9D9D9", end_color="D9D9D9", fill_type="solid")

# Data Rows
for row_num, row_data in enumerate(data_classification, 4):
    for col_num, val in enumerate(row_data, 1):
        cell = ws.cell(row=row_num, column=col_num, value=val)
        if col_num > 1:
            cell.number_format = "0.000"

# Question Analysis
ws.cell(row=13, column=1, value="Which model gave you the best result and which was the worst?").font = Font(bold=True)
ws.cell(row=14, column=1, value="Best: Bayesian Ensemble / XGBoost (ROC-AUC: 0.991, F1: 0.940). Worst: K-Nearest Neighbors (F1: 0.806).")

# Save file
wb.save("Model_Discussion_Guidelines_Filled.xlsx")
print("Successfully generated 'Model_Discussion_Guidelines_Filled.xlsx'!")

Successfully generated 'Model_Discussion_Guidelines_Filled.xlsx'!
